# Pipeline Bronze : JSON → Delta Lake

## Objectif
Implémenter une pipeline permettant de :
1. Lire un flux de données au format JSON
2. Appliquer des transformations basiques (nettoyage, filtrage, projection)
3. Écrire les données dans une table Delta (niveau Bronze)
4. Configurer une stratégie de tolérance aux pannes via checkpointing

## Contexte SmartTech
Les données proviennent de capteurs IoT installés dans des bâtiments intelligents :
- Température, humidité, consommation d'énergie
- Détection d'anomalies
- Informations de localisation


## 1. Configuration de l'environnement Spark


In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *
import os

# Les variables d'environnement sont injectées par docker-compose depuis le fichier .env
# Pas besoin de load_dotenv() car docker-compose passe les variables directement au conteneur

# Configuration des chemins depuis les variables d'environnement (avec valeurs par défaut)
DATA_DIR = os.getenv("DATA_DIR", "/opt/spark/data")
DELTA_BRONZE_PATH = os.getenv("DELTA_BRONZE_PATH", "/opt/spark/delta/bronze")
CHECKPOINT_PATH = os.getenv("CHECKPOINT_BRONZE_PATH", "/opt/spark/checkpoints/bronze")

# Configuration Spark depuis les variables d'environnement
SPARK_APP_NAME = os.getenv("SPARK_APP_NAME", "SmartTech-Bronze-Pipeline")

# Créer la session Spark avec support Delta Lake
# IMPORTANT : delta-spark==2.4.0 est installé via pip dans le Dockerfile
# configure_spark_with_delta_pip() utilise les JARs déjà installés par pip (pas de téléchargement Maven)
builder = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Utiliser configure_spark_with_delta_pip() qui utilise les JARs installés par pip (delta-spark==2.4.0)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("✓ Spark Session créée avec succès")
print(f"✓ Version Spark : {spark.version}")
print(f"✓ DATA_DIR : {DATA_DIR}")
print(f"✓ DELTA_BRONZE_PATH : {DELTA_BRONZE_PATH}")
print(f"✓ CHECKPOINT_PATH : {CHECKPOINT_PATH}")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2efbfcbf-a079-4193-9f6f-440166da614e;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (248ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (67ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (86ms)
:: resolution report :: resolve 2117ms :: artifacts dl 404ms
	:

✓ Spark Session créée avec succès
✓ Version Spark : 3.4.0
✓ DATA_DIR : /opt/spark/data
✓ DELTA_BRONZE_PATH : /opt/spark/delta/bronze
✓ CHECKPOINT_PATH : /opt/spark/checkpoints/bronze


## 2. Définition du schéma des données


In [2]:
# Schéma des données IoT SmartTech
sensor_schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("energy_consumption", DoubleType(), True),
    StructField("anomaly_detected", BooleanType(), True),
    StructField("building_id", StringType(), True),
    StructField("sensor_type", StringType(), True),
    StructField("location", StringType(), True)
])

print("✓ Schéma défini pour les données IoT")


✓ Schéma défini pour les données IoT


## 3. Lecture du flux JSON


In [3]:
# Configuration de la source de lecture JSON
# maxFilesPerTrigger : nombre de fichiers traités par micro-batch
raw_stream = spark \
    .readStream \
    .format("json") \
    .schema(sensor_schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiline", "true") \
    .load(DATA_DIR)

print("✓ Source de lecture JSON configurée")
print(f"✓ Schéma de la source :")
raw_stream.printSchema()


✓ Source de lecture JSON configurée
✓ Schéma de la source :
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- energy_consumption: double (nullable = true)
 |-- anomaly_detected: boolean (nullable = true)
 |-- building_id: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- location: string (nullable = true)



## 4. Transformations basiques

### 4.1 Nettoyage des données


In [4]:
# Nettoyage et validation des données
# IMPORTANT : Gestion robuste du format ISO 8601
# Normalise les timestamps pour gérer les deux cas :
# 1. Avec 'Z' (UTC) : '2025-01-12T08:22:04Z' → '2025-01-12T08:22:04+00:00'
# 2. Sans timezone : '2025-01-12T08:22:04' → '2025-01-12T08:22:04+00:00'
# 3. Avec offset : '2025-01-12T08:22:04+02:00' → garde tel quel
cleaned_stream = raw_stream \
    .withColumn("timestamp_normalized", 
        # Normaliser le timestamp : remplacer 'Z' par '+00:00' OU ajouter '+00:00' si absent
        when(
            col("timestamp").rlike(".*[+-]\\d{2}:\\d{2}$"),  # Déjà un timezone offset
            col("timestamp")
        ).when(
            col("timestamp").rlike(".*Z$"),  # Format avec 'Z'
            regexp_replace(col("timestamp"), "Z$", "+00:00")
        ).otherwise(
            # Pas de timezone : ajouter '+00:00' (assume UTC)
            concat(col("timestamp"), lit("+00:00"))
        )
    ) \
    .withColumn("timestamp", 
        # Parser avec le format ISO 8601 qui supporte le timezone offset
        # Format : yyyy-MM-dd'T'HH:mm:ss[+/-]HH:mm
        to_timestamp(
            col("timestamp_normalized"), 
            "yyyy-MM-dd'T'HH:mm:ssXXX"
        )
    ) \
    .drop("timestamp_normalized") \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("temperature", 
        when(col("temperature").isNull(), 20.0).otherwise(col("temperature"))
    ) \
    .withColumn("humidity", 
        when(col("humidity").isNull(), 50.0).otherwise(col("humidity"))
    ) \
    .withColumn("energy_consumption", 
        when(col("energy_consumption").isNull(), 0.0).otherwise(col("energy_consumption"))
    )

print("✓ Nettoyage des valeurs nulles effectué")


✓ Nettoyage des valeurs nulles effectué


### 4.2 Filtrage des données invalides


In [5]:
# Filtrage : garder uniquement les données valides
# Validation des plages de valeurs réalistes
filtered_stream = cleaned_stream \
    .filter(
        (col("sensor_id").isNotNull()) &
        (col("timestamp").isNotNull()) &
        (col("temperature") >= -50) & (col("temperature") <= 60) &  # Plage réaliste
        (col("humidity") >= 0) & (col("humidity") <= 100) &  # Humidité en pourcentage
        (col("energy_consumption") >= 0)  # Consommation positive
    )

print("✓ Filtrage des données invalides configuré")


✓ Filtrage des données invalides configuré


### 4.3 Projection et ajout de colonnes calculées


In [6]:
# Projection : sélectionner et enrichir les colonnes
bronze_stream = filtered_stream \
    .select(
        col("sensor_id"),
        col("timestamp"),
        col("temperature"),
        col("humidity"),
        col("energy_consumption"),
        col("anomaly_detected"),
        col("building_id"),
        col("sensor_type"),
        col("location"),
        col("ingestion_timestamp"),
        # Colonnes calculées
        when(col("temperature") > 30, "HIGH").otherwise("NORMAL").alias("temp_status"),
        when(col("energy_consumption") > 800, "HIGH").otherwise("NORMAL").alias("energy_status")
    )

print("✓ Projection et enrichissement des données effectués")
print(f"✓ Schéma final Bronze :")
bronze_stream.printSchema()


✓ Projection et enrichissement des données effectués
✓ Schéma final Bronze :
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- energy_consumption: double (nullable = true)
 |-- anomaly_detected: boolean (nullable = true)
 |-- building_id: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- location: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- temp_status: string (nullable = false)
 |-- energy_status: string (nullable = false)



## 5. Écriture dans Delta Lake (Niveau Bronze)


In [7]:
# Configuration de l'écriture vers Delta Lake
# Mode append : ajoute uniquement les nouvelles lignes
query = bronze_stream \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("path", DELTA_BRONZE_PATH) \
    .partitionBy("building_id", "sensor_type") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✓ Pipeline Bronze démarrée")
print(f"✓ Écriture dans : {DELTA_BRONZE_PATH}")
print(f"✓ Checkpoint dans : {CHECKPOINT_PATH}")
print(f"✓ Trigger : toutes les 10 secondes")


✓ Pipeline Bronze démarrée
✓ Écriture dans : /opt/spark/delta/bronze
✓ Checkpoint dans : /opt/spark/checkpoints/bronze
✓ Trigger : toutes les 10 secondes


25/12/15 14:26:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 6. Monitoring de la pipeline


In [8]:
# Attendre quelques micro-batches pour voir les données
import time

print("Pipeline en cours d'exécution...")
print("Attente de 30 secondes pour traiter les données...")

time.sleep(30)

# Vérifier le statut
print(f"\n✓ Statut de la query : {query.status}")
print(f"✓ Dernière progression : {query.lastProgress}")


Pipeline en cours d'exécution...
Attente de 30 secondes pour traiter les données...


25/12/15 14:26:05 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                


✓ Statut de la query : {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}
✓ Dernière progression : {'id': 'b7e6c97a-8d37-4ecb-8d34-52b8485cff92', 'runId': '9212618b-fc1d-454c-97c9-1e1f0470c73b', 'name': None, 'timestamp': '2025-12-15T14:26:30.000Z', 'batchId': 3, 'numInputRows': 1, 'inputRowsPerSecond': 0.1, 'processedRowsPerSecond': 0.6082725060827251, 'durationMs': {'addBatch': 1057, 'commitOffsets': 168, 'getBatch': 51, 'latestOffset': 199, 'queryPlanning': 17, 'triggerExecution': 1644, 'walCommit': 150}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[file:/opt/spark/data]', 'startOffset': {'logOffset': 2}, 'endOffset': {'logOffset': 3}, 'latestOffset': None, 'numInputRows': 1, 'inputRowsPerSecond': 0.1, 'processedRowsPerSecond': 0.6082725060827251}], 'sink': {'description': 'DeltaSink[/opt/spark/delta/bronze]', 'numOutputRows': -1}}


## 7. Vérification des données écrites


In [9]:
# Lire les données Delta Lake pour vérification
bronze_df = spark.read.format("delta").load(DELTA_BRONZE_PATH)

print(f"✓ Nombre total d'enregistrements dans Bronze : {bronze_df.count()}")
print("\n✓ Aperçu des données :")
bronze_df.show(10, truncate=False)

print("\n✓ Statistiques par building :")
bronze_df.groupBy("building_id").count().show()

print("\n✓ Statistiques par type de capteur :")
bronze_df.groupBy("sensor_type").count().show()


✓ Nombre total d'enregistrements dans Bronze : 0

✓ Aperçu des données :
+---------+---------+-----------+--------+------------------+----------------+-----------+-----------+--------+-------------------+-----------+-------------+
|sensor_id|timestamp|temperature|humidity|energy_consumption|anomaly_detected|building_id|sensor_type|location|ingestion_timestamp|temp_status|energy_status|
+---------+---------+-----------+--------+------------------+----------------+-----------+-----------+--------+-------------------+-----------+-------------+
+---------+---------+-----------+--------+------------------+----------------+-----------+-----------+--------+-------------------+-----------+-------------+


✓ Statistiques par building :
+-----------+-----+
|building_id|count|
+-----------+-----+
+-----------+-----+


✓ Statistiques par type de capteur :
+-----------+-----+
|sensor_type|count|
+-----------+-----+
+-----------+-----+



## 8. Test de tolérance aux pannes


In [10]:
# Arrêter la query pour simuler une panne
query.stop()
print("✓ Pipeline arrêtée (simulation de panne)")

# Relancer la pipeline - elle devrait reprendre depuis le checkpoint
print("\n✓ Redémarrage de la pipeline depuis le checkpoint...")

query_restart = bronze_stream \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("path", DELTA_BRONZE_PATH) \
    .partitionBy("building_id", "sensor_type") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✓ Pipeline redémarrée - les données seront reprises depuis le dernier offset traité")


✓ Pipeline arrêtée (simulation de panne)

✓ Redémarrage de la pipeline depuis le checkpoint...
✓ Pipeline redémarrée - les données seront reprises depuis le dernier offset traité


25/12/15 14:26:35 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 9. Arrêt propre de la pipeline


In [11]:
# Arrêter la pipeline proprement
query_restart.stop()
print("✓ Pipeline arrêtée proprement")

# Afficher un résumé final
final_count = spark.read.format("delta").load(DELTA_BRONZE_PATH).count()
print(f"\n✓ Total d'enregistrements dans la table Bronze : {final_count}")
print("\n✓ Pipeline Bronze terminée avec succès !")


✓ Pipeline arrêtée proprement

✓ Total d'enregistrements dans la table Bronze : 0

✓ Pipeline Bronze terminée avec succès !


## Résumé de la pipeline Bronze

### Ce qui a été implémenté :

1. ✅ **Lecture du flux JSON** : Configuration de la source avec schéma défini
2. ✅ **Transformations** :
   - Nettoyage des valeurs nulles
   - Validation des plages de valeurs
   - Filtrage des données invalides
   - Enrichissement avec colonnes calculées
3. ✅ **Écriture Delta Lake** : Niveau Bronze avec partitionnement
4. ✅ **Checkpointing** : Tolérance aux pannes configurée
5. ✅ **Monitoring** : Suivi de l'exécution et vérification des données

### Points clés :
- **Mode append** : Ajoute uniquement les nouvelles lignes
- **Partitionnement** : Par `building_id` et `sensor_type` pour optimiser les requêtes
- **Trigger** : ProcessingTime de 10 secondes
- **Checkpoint** : Permet la reprise après panne sans perte de données
